# OPENBREWERY - CONFIG NOTEBOOK

# 1. GENERAL CONFIGURATIONS

## 1.1 Catalog

In [0]:
catalog_name = "brewing"

## 1.2 Storage Account

In [0]:
storage_account = "sabeesdevwestus001"

## 1.3 Containers

In [0]:
landing_container = "bees-landing-dev-001"
bronze_container = "bees-bronze-dev-001"
silver_container = "bees-silver-dev-001"
gold_container = "bees-gold-dev-001"
config_container = "bees-config-dev-001"

## 1.4 Paths - Volumes

In [0]:
landing_volume_path = (
    f"abfss://{landing_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/landing/"
)

config_volume_path = (
    f"abfss://{config_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/config/"
)

## 1.5 Paths - Tables

In [0]:
bronze_table_path = (
    f"abfss://{bronze_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/bronze/tables/openbrewery_bronze"
)

silver_table_path = (
    f"abfss://{silver_container}@"
    f"{storage_account}.dfs.core.windows.net/"
    "openbrewery/silver/tables/openbrewery_silver"
)

## 1.6 Variables

In [0]:
landing_volume = (
    f"/Volumes/{catalog_name}/landing/"
    "openbrewery_landing_volume"
)

config_volume = (
    f"/Volumes/{catalog_name}/config/"
    "openbrewery_config_volume"
)

schema_location = (
    f"{config_volume}/schemas/openbrewery"
)

checkpoint_path = (
    f"{config_volume}/checkpoints/openbrewery"
)

bronze_table = (
    f"{catalog_name}.bronze.openbrewery_bronze"
)

silver_table = (
    f"{catalog_name}.silver.openbrewery_silver"
)

control_table = (
    f"{catalog_name}.control.openbrewery_control"
)

monitoring_schema = "control"

monitoring_table = (
    f"{catalog_name}.control.pipeline_monitoring"
)

data_quality_table = (
    f"{catalog_name}.control.data_quality"
)

quarantine_silver_table = (
    f"{catalog_name}.control.quarantine_silver"
)

# 2. Creating Architecture

## 1.1 Catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS brewing;

## 1.2 Schemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS brewing.landing;
CREATE SCHEMA IF NOT EXISTS brewing.bronze;
CREATE SCHEMA IF NOT EXISTS brewing.silver;
CREATE SCHEMA IF NOT EXISTS brewing.gold;
CREATE SCHEMA IF NOT EXISTS brewing.config;
CREATE SCHEMA IF NOT EXISTS brewing.control;

## 1.3 External Volumes

### 1.3.1 Config

In [0]:
spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS
{catalog_name}.config.openbrewery_config_volume
LOCATION '{config_volume_path}'
""")

### 1.3.2 Landing

In [0]:
spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS
{catalog_name}.landing.openbrewery_landing_volume
LOCATION '{landing_volume_path}'
""")

## 1.4 Tables

## 1.4.1 Monitoring Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS
{catalog_name}.control.pipeline_monitoring (

    execution_id STRING,

    pipeline_name STRING,

    layer STRING,

    start_time TIMESTAMP,

    end_time TIMESTAMP,

    status STRING,

    records_processed BIGINT,

    error_message STRING,

    created_at TIMESTAMP

)

USING DELTA

""")

### 1.4.2 Bronze Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{bronze_table}
(

    id STRING,
    name STRING,
    brewery_type STRING,

    address_1 STRING,
    address_2 STRING,
    address_3 STRING,

    city STRING,
    state STRING,
    state_province STRING,
    postal_code STRING,
    country STRING,

    longitude DOUBLE,
    latitude DOUBLE,

    phone STRING,
    website_url STRING,
    street STRING,

    metadata STRUCT<
        ingestion_timestamp_utc: STRING,
        page: BIGINT,
        records: BIGINT,
        source: STRING
    >,

    _rescued_data STRING,

    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE,

    source_file STRING,
    source_path STRING,

    file_modification_time TIMESTAMP,
    file_size BIGINT,

    execution_id STRING,

    record_hash STRING

)
USING DELTA
PARTITIONED BY (ingestion_date)
LOCATION '{bronze_table_path}'
""")

### 1.4.3 Silver Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{silver_table}
(

    id STRING,
    name STRING,
    brewery_type STRING,

    address_1 STRING,
    address_2 STRING,
    address_3 STRING,

    city STRING,
    state STRING,
    state_province STRING,
    postal_code STRING,
    country STRING,

    longitude DOUBLE,
    latitude DOUBLE,

    phone STRING,
    website_url STRING,
    street STRING,

    record_hash STRING,

    valid_from TIMESTAMP,
    valid_to TIMESTAMP,

    is_current BOOLEAN,

    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE

)
USING DELTA
LOCATION '{silver_table_path}'
""")

### 1.4.4 Gold Dim Brewery Type Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS brewing.gold.dim_brewery_type
(
    brewery_type_sk STRING,
    brewery_type STRING
)

USING DELTA;

### 1.4.5 Gold Dim Location Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS brewing.gold.dim_location
(
    location_sk STRING,
    city STRING,
    state STRING,
    country STRING
)

USING DELTA;

### 1.4.6 Gold Dim Brewery Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS brewing.gold.dim_brewery
(
    brewery_sk STRING,
    brewery_id STRING,
    brewery_name STRING,
    brewery_type_sk STRING,
    location_sk STRING,
    website_url STRING,
    phone STRING,
    ingestion_timestamp TIMESTAMP
)

USING DELTA;

### 1.4.7 Gold Fact Brewery Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS brewing.gold.fact_brewery
(
    snapshot_date DATE,
    brewery_sk STRING,
    brewery_type_sk STRING,
    location_sk STRING,
    brewery_count INT,
    ingestion_timestamp TIMESTAMP
)

USING DELTA

PARTITIONED BY (snapshot_date);

### 1.4.8 Gold Control Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS
{gold_control_table}

(
    pipeline_name STRING,
    last_ingestion_timestamp TIMESTAMP,
    processed_at TIMESTAMP,
    records_processed BIGINT
)

USING DELTA

""")

### 1.4.9 Control Table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{control_table}
(

    pipeline_name STRING,
    last_ingestion_timestamp TIMESTAMP,
    processed_at TIMESTAMP,
    records_processed BIGINT

)
USING DELTA
""")

### 1.4.10 Data Quality Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS
{catalog_name}.control.data_quality

(

    execution_id STRING,

    pipeline_name STRING,

    layer STRING,

    table_name STRING,

    rule_name STRING,

    invalid_records BIGINT,

    created_at TIMESTAMP

)

USING DELTA

""")

### 1.4.11 Quarentine Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS
{catalog_name}.control.quarantine_silver

(

    id STRING,

    name STRING,

    brewery_type STRING,

    address_1 STRING,

    address_2 STRING,

    address_3 STRING,

    city STRING,

    state STRING,

    state_province STRING,

    postal_code STRING,

    country STRING,

    longitude DOUBLE,

    latitude DOUBLE,

    phone STRING,

    website_url STRING,

    street STRING,

    metadata STRUCT<
        ingestion_timestamp_utc: STRING,
        page: BIGINT,
        records: BIGINT,
        source: STRING
    >,

    _rescued_data STRING,

    ingestion_timestamp TIMESTAMP,

    ingestion_date DATE,

    source_file STRING,

    source_path STRING,

    file_modification_time TIMESTAMP,

    file_size BIGINT,

    execution_id STRING,

    record_hash STRING,

    dq_rule STRING,

    quarantine_timestamp TIMESTAMP

)

USING DELTA

""")

## 1.5 Table Schemas

### 1.5.1 Data Quality

In [0]:
dq_schema = StructType([

    StructField(
        "execution_id",
        StringType(),
        True
    ),

    StructField(
        "pipeline_name",
        StringType(),
        True
    ),

    StructField(
        "layer",
        StringType(),
        True
    ),

    StructField(
        "table_name",
        StringType(),
        True
    ),

    StructField(
        "rule_name",
        StringType(),
        True
    ),

    StructField(
        "invalid_records",
        LongType(),
        True
    ),

    StructField(
        "created_at",
        TimestampType(),
        True
    )
])

### 1.5.2 Monitoring

In [0]:
monitoring_schema = StructType([

    StructField(
        "execution_id",
        StringType(),
        True
    ),

    StructField(
        "pipeline_name",
        StringType(),
        True
    ),

    StructField(
        "layer",
        StringType(),
        True
    ),

    StructField(
        "start_time",
        TimestampType(),
        True
    ),

    StructField(
        "end_time",
        TimestampType(),
        True
    ),

    StructField(
        "status",
        StringType(),
        True
    ),

    StructField(
        "records_processed",
        LongType(),
        True
    ),

    StructField(
        "error_message",
        StringType(),
        True
    ),

    StructField(
        "created_at",
        TimestampType(),
        True
    )
])

### 1.5.3 Control

In [0]:
    control_schema = StructType([

        StructField(
            "pipeline_name",
            StringType(),
            True
        ),

        StructField(
            "last_ingestion_timestamp",
            TimestampType(),
            True
        ),

        StructField(
            "processed_at",
            TimestampType(),
            True
        ),

        StructField(
            "records_processed",
            LongType(),
            True
        )
    ])


In [0]:
    log_schema = StructType([

        StructField(
            "execution_id",
            StringType(),
            True
        ),

        StructField(
            "pipeline_name",
            StringType(),
            True
        ),

        StructField(
            "layer",
            StringType(),
            True
        ),

        StructField(
            "step_name",
            StringType(),
            True
        ),

        StructField(
            "start_time",
            TimestampType(),
            True
        ),

        StructField(
            "end_time",
            TimestampType(),
            True
        ),

        StructField(
            "duration_seconds",
            DoubleType(),
            True
        ),

        StructField(
            "status",
            StringType(),
            True
        ),

        StructField(
            "records_read",
            LongType(),
            True
        ),

        StructField(
            "records_written",
            LongType(),
            True
        ),

        StructField(
            "error_message",
            StringType(),
            True
        ),

        StructField(
            "created_at",
            TimestampType(),
            True
        )
    ])
